# Chapter 12.7: Mixture-of-Experts Serving Optimization

## Overview

Mixture-of-Experts (MoE) models like Mixtral, DeepSeek-V2/V3, and Grok use sparse expert layers to scale model capacity without proportionally scaling compute. However, serving MoE models introduces unique challenges.

### Key Challenges
1. **Expert load imbalance**: Some experts are activated more than others
2. **Large memory footprint**: All experts must be loadable, even if only few are active
3. **Expert communication**: With expert parallelism, tokens must be routed across GPUs
4. **Dynamic computation**: Different tokens activate different experts

### Learning Objectives
1. Profile expert utilization patterns
2. Implement expert caching with LRU eviction
3. Simulate expert parallelism strategies
4. Build expert load balancers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part12/chapter_12.7_moe_optimization.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part12/chapter_12.7_moe_optimization.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict, OrderedDict
import time

np.random.seed(42)
plt.style.use('default')
print("MoE optimization environment ready.")

## 1. MoE Architecture Overview

In [ ]:
@dataclass
class MoEConfig:
    name: str
    num_layers: int
    hidden_dim: int
    num_experts: int
    top_k: int  # Number of experts activated per token
    expert_dim: int  # FFN intermediate dim per expert
    num_shared_experts: int = 0  # DeepSeek-style shared experts
    param_bytes: int = 2  # FP16

    @property
    def total_params_billions(self):
        # Attention params per layer
        attn_params = 4 * self.hidden_dim * self.hidden_dim  # Q, K, V, O
        # Expert FFN params per layer
        expert_params = self.num_experts * 3 * self.hidden_dim * self.expert_dim  # up, gate, down
        # Shared expert params
        shared_params = self.num_shared_experts * 3 * self.hidden_dim * self.expert_dim
        total = self.num_layers * (attn_params + expert_params + shared_params)
        return total / 1e9

    @property
    def active_params_billions(self):
        attn_params = 4 * self.hidden_dim * self.hidden_dim
        # Only top_k experts active per token
        active_expert_params = self.top_k * 3 * self.hidden_dim * self.expert_dim
        shared_params = self.num_shared_experts * 3 * self.hidden_dim * self.expert_dim
        total = self.num_layers * (attn_params + active_expert_params + shared_params)
        return total / 1e9

    @property
    def expert_size_gb(self):
        """Size of a single expert (all layers)."""
        params_per_expert = self.num_layers * 3 * self.hidden_dim * self.expert_dim
        return params_per_expert * self.param_bytes / 1e9

    @property
    def total_size_gb(self):
        return self.total_params_billions * 1e9 * self.param_bytes / 1e9


models = {
    'Mixtral-8x7B': MoEConfig('Mixtral-8x7B', 32, 4096, 8, 2, 14336),
    'Mixtral-8x22B': MoEConfig('Mixtral-8x22B', 56, 6144, 8, 2, 16384),
    'DeepSeek-V2': MoEConfig('DeepSeek-V2', 60, 5120, 160, 6, 1536, num_shared_experts=2),
    'DeepSeek-V3': MoEConfig('DeepSeek-V3', 61, 7168, 256, 8, 2048, num_shared_experts=1),
    'Grok-1': MoEConfig('Grok-1', 64, 6144, 8, 2, 32768),
}

print(f"{'Model':>18} {'Total Params':>13} {'Active Params':>14} {'Experts':>8} "
      f"{'Top-K':>6} {'Total Size':>10} {'Expert Size':>11}")
print("-" * 85)
for name, m in models.items():
    print(f"{name:>18} {m.total_params_billions:>11.1f}B {m.active_params_billions:>12.1f}B "
          f"{m.num_experts:>8} {m.top_k:>6} {m.total_size_gb:>8.1f}GB {m.expert_size_gb:>9.2f}GB")

## 2. Demo: Profile Expert Utilization Patterns

In [ ]:
class ExpertRouter:
    """Simulate expert routing with various strategies."""

    def __init__(self, num_experts: int, top_k: int, routing: str = 'learned'):
        self.num_experts = num_experts
        self.top_k = top_k
        self.routing = routing
        # Expert popularity bias (some experts naturally more popular)
        self.popularity = np.random.dirichlet(np.ones(num_experts) * 2)
        self.call_counts = np.zeros(num_experts, dtype=int)

    def route(self, num_tokens: int) -> np.ndarray:
        """Route tokens to experts. Returns [num_tokens, top_k] expert indices."""
        if self.routing == 'uniform':
            # Random uniform routing
            indices = np.array([np.random.choice(self.num_experts, self.top_k, replace=False)
                               for _ in range(num_tokens)])
        elif self.routing == 'learned':
            # Biased routing (simulates learned router)
            indices = np.array([
                np.random.choice(self.num_experts, self.top_k, replace=False,
                                p=self.popularity)
                for _ in range(num_tokens)
            ])
        elif self.routing == 'skewed':
            # Heavily skewed (pathological case)
            skewed_pop = np.zeros(self.num_experts)
            skewed_pop[:3] = 0.8 / 3  # 80% to top 3 experts
            skewed_pop[3:] = 0.2 / (self.num_experts - 3)
            indices = np.array([
                np.random.choice(self.num_experts, self.top_k, replace=False,
                                p=skewed_pop)
                for _ in range(num_tokens)
            ])
        else:
            raise ValueError(f"Unknown routing: {self.routing}")

        # Update counts
        for idx_row in indices:
            for idx in idx_row:
                self.call_counts[idx] += 1

        return indices

    def load_balance_score(self) -> float:
        """Compute load balance score (1.0 = perfect, higher = more imbalanced)."""
        if np.sum(self.call_counts) == 0:
            return 1.0
        freq = self.call_counts / np.sum(self.call_counts)
        ideal_freq = 1.0 / self.num_experts
        return np.sum(freq * self.num_experts * freq) / (self.top_k / self.num_experts)


# Profile different routing strategies
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

config = models['Mixtral-8x7B']
num_tokens = 10000

for col, routing in enumerate(['uniform', 'learned', 'skewed']):
    router = ExpertRouter(config.num_experts, config.top_k, routing)
    indices = router.route(num_tokens)

    # Expert activation frequency
    ax = axes[0, col]
    ax.bar(range(config.num_experts), router.call_counts, color='steelblue')
    ideal = num_tokens * config.top_k / config.num_experts
    ax.axhline(ideal, color='red', linestyle='--', alpha=0.5, label=f'Ideal ({ideal:.0f})')
    ax.set_xlabel('Expert ID')
    ax.set_ylabel('Activation Count')
    ax.set_title(f'{routing.title()} Routing\nBalance={router.load_balance_score():.2f}')
    ax.legend()

    # Co-activation heatmap
    ax = axes[1, col]
    coactivation = np.zeros((config.num_experts, config.num_experts))
    for token_experts in indices:
        for i in range(len(token_experts)):
            for j in range(len(token_experts)):
                coactivation[token_experts[i], token_experts[j]] += 1
    im = ax.imshow(coactivation, cmap='hot')
    ax.set_xlabel('Expert')
    ax.set_ylabel('Expert')
    ax.set_title(f'Co-activation ({routing})')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('expert_utilization.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Profile DeepSeek-V3 style with many experts
deepseek_config = models['DeepSeek-V3']
router = ExpertRouter(deepseek_config.num_experts, deepseek_config.top_k, 'learned')
indices = router.route(50000)

print(f"DeepSeek-V3 Expert Utilization ({deepseek_config.num_experts} experts, top-{deepseek_config.top_k}):")
print(f"  Load balance score: {router.load_balance_score():.2f} (1.0 = perfect)")

# Token-expert distribution
sorted_counts = np.sort(router.call_counts)[::-1]
print(f"  Most active expert: {sorted_counts[0]} calls")
print(f"  Least active expert: {sorted_counts[-1]} calls")
print(f"  Top 10% experts handle: {np.sum(sorted_counts[:deepseek_config.num_experts//10]) / np.sum(sorted_counts):.1%} of traffic")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(deepseek_config.num_experts), sorted_counts, width=1.0, alpha=0.7)
ax1.set_xlabel('Expert (sorted by popularity)')
ax1.set_ylabel('Activation Count')
ax1.set_title(f'Expert Popularity Distribution (n={deepseek_config.num_experts})')

# CDF of expert activations
cumulative = np.cumsum(sorted_counts) / np.sum(sorted_counts)
ax2.plot(np.arange(1, len(cumulative) + 1) / len(cumulative), cumulative, linewidth=2)
ax2.axhline(0.8, color='red', linestyle='--', alpha=0.5)
ax2.axhline(0.9, color='orange', linestyle='--', alpha=0.5)
idx_80 = np.searchsorted(cumulative, 0.8) + 1
idx_90 = np.searchsorted(cumulative, 0.9) + 1
ax2.set_xlabel('Fraction of Experts')
ax2.set_ylabel('Fraction of Activations')
ax2.set_title(f'80% coverage: top {idx_80} experts, 90%: top {idx_90}')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('deepseek_experts.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Demo: Expert Caching with LRU

In [ ]:
class ExpertCache:
    """LRU cache for expert weights on GPU."""

    def __init__(self, num_total_experts: int, cache_capacity: int,
                 expert_size_gb: float, load_bandwidth_gbps: float = 64.0):
        self.num_experts = num_total_experts
        self.capacity = cache_capacity
        self.expert_size = expert_size_gb
        self.bandwidth = load_bandwidth_gbps

        self.cache = OrderedDict()  # expert_id -> True (LRU order)
        self.hits = 0
        self.misses = 0
        self.total_load_time = 0  # seconds
        self.load_events = []  # Track individual loads

    def access(self, expert_id: int) -> float:
        """Access an expert. Returns load time if cache miss."""
        if expert_id in self.cache:
            self.cache.move_to_end(expert_id)
            self.hits += 1
            return 0.0

        self.misses += 1
        load_time = self.expert_size / self.bandwidth
        self.total_load_time += load_time
        self.load_events.append({'expert': expert_id, 'time': load_time})

        # Evict if full
        while len(self.cache) >= self.capacity:
            self.cache.popitem(last=False)  # Remove LRU

        self.cache[expert_id] = True
        return load_time

    def access_batch(self, expert_ids: List[int]) -> float:
        """Access multiple experts. Returns total overhead."""
        total_overhead = 0
        for eid in expert_ids:
            total_overhead += self.access(eid)
        return total_overhead

    @property
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0

    def stats(self):
        return {
            'hit_rate': self.hit_rate,
            'hits': self.hits,
            'misses': self.misses,
            'total_load_time_s': self.total_load_time,
            'cache_size': len(self.cache),
            'cache_memory_gb': len(self.cache) * self.expert_size,
        }


# Simulate expert caching for DeepSeek-V3
config = models['DeepSeek-V3']
cache_sizes = [16, 32, 64, 128, 256]  # Number of experts cached on GPU

# Generate workload
router = ExpertRouter(config.num_experts, config.top_k, 'learned')
workload_tokens = 100000
all_indices = router.route(workload_tokens)

results = {}
for cache_size in cache_sizes:
    cache = ExpertCache(config.num_experts, cache_size,
                         config.expert_size_gb, load_bandwidth_gbps=64.0)

    for token_experts in all_indices:
        cache.access_batch(token_experts.tolist())

    results[cache_size] = cache.stats()
    print(f"Cache size={cache_size:>4}: hit_rate={results[cache_size]['hit_rate']:.3f}, "
          f"load_time={results[cache_size]['total_load_time_s']:.1f}s, "
          f"memory={results[cache_size]['cache_memory_gb']:.1f}GB")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(cache_sizes, [results[cs]['hit_rate'] for cs in cache_sizes], 'bo-', linewidth=2)
ax1.set_xlabel('Cache Size (experts)')
ax1.set_ylabel('Hit Rate')
ax1.set_title(f'Expert Cache Hit Rate (n={config.num_experts})')
ax1.grid(True, alpha=0.3)
ax1.axhline(1.0, color='green', linestyle='--', alpha=0.3)

ax2.plot(cache_sizes, [results[cs]['total_load_time_s'] for cs in cache_sizes], 'ro-', linewidth=2)
ax2.set_xlabel('Cache Size (experts)')
ax2.set_ylabel('Total Load Time (s)')
ax2.set_title('Expert Loading Overhead')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('expert_cache.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Demo: Expert Parallelism Simulation

In [ ]:
class ExpertParallelismSimulator:
    """Simulate different expert parallelism strategies."""

    def __init__(self, config: MoEConfig, num_gpus: int,
                 gpu_compute_tflops: float = 989,
                 interconnect_gbps: float = 900):
        self.config = config
        self.num_gpus = num_gpus
        self.compute = gpu_compute_tflops
        self.interconnect = interconnect_gbps

    def expert_parallel(self, batch_size: int, seq_len: int) -> dict:
        """Expert Parallelism: experts distributed across GPUs."""
        experts_per_gpu = self.config.num_experts // self.num_gpus

        # All-to-all: tokens sent to their assigned expert's GPU
        tokens_per_expert = batch_size * seq_len * self.config.top_k / self.config.num_experts
        # Each token sends hidden_dim * param_bytes
        bytes_per_token = self.config.hidden_dim * self.config.param_bytes
        total_comm_bytes = batch_size * seq_len * self.config.top_k * bytes_per_token
        # Two all-to-all: dispatch + combine
        comm_time = 2 * total_comm_bytes / (self.interconnect * 1e9)

        # Compute: each GPU processes its experts
        expert_flops = 2 * 3 * self.config.hidden_dim * self.config.expert_dim  # up+gate+down
        flops_per_gpu = tokens_per_expert * experts_per_gpu * expert_flops
        compute_time = flops_per_gpu / (self.compute * 1e12)

        # Load imbalance penalty
        imbalance = 1.2  # 20% overhead from load imbalance

        total_time = compute_time * imbalance + comm_time

        return {
            'strategy': 'expert_parallel',
            'total_time_ms': total_time * 1000,
            'compute_time_ms': compute_time * imbalance * 1000,
            'comm_time_ms': comm_time * 1000,
            'experts_per_gpu': experts_per_gpu,
            'memory_per_gpu_gb': experts_per_gpu * self.config.expert_size_gb / self.config.num_layers,
        }

    def tensor_parallel(self, batch_size: int, seq_len: int) -> dict:
        """Tensor Parallelism: each expert split across GPUs."""
        # Each GPU has 1/N of every expert
        tokens = batch_size * seq_len

        # Compute: distributed across GPUs
        expert_flops = 2 * 3 * self.config.hidden_dim * self.config.expert_dim * self.config.top_k
        total_flops = tokens * expert_flops
        flops_per_gpu = total_flops / self.num_gpus
        compute_time = flops_per_gpu / (self.compute * 1e12)

        # All-reduce after each expert layer
        bytes_per_allreduce = tokens * self.config.hidden_dim * self.config.param_bytes
        comm_time = 2 * bytes_per_allreduce * (self.num_gpus - 1) / self.num_gpus / (self.interconnect * 1e9)

        total_time = compute_time + comm_time

        return {
            'strategy': 'tensor_parallel',
            'total_time_ms': total_time * 1000,
            'compute_time_ms': compute_time * 1000,
            'comm_time_ms': comm_time * 1000,
            'memory_per_gpu_gb': self.config.total_size_gb / self.num_gpus,
        }

    def hybrid_parallel(self, batch_size: int, seq_len: int,
                         expert_groups: int = 2) -> dict:
        """Hybrid: Expert parallel between groups, tensor parallel within."""
        gpus_per_group = self.num_gpus // expert_groups
        experts_per_group = self.config.num_experts // expert_groups

        tokens = batch_size * seq_len
        tokens_per_expert = tokens * self.config.top_k / self.config.num_experts

        # Inter-group all-to-all (expert routing)
        bytes_per_token = self.config.hidden_dim * self.config.param_bytes
        intergroup_bytes = tokens * self.config.top_k * bytes_per_token / expert_groups
        intergroup_time = 2 * intergroup_bytes / (self.interconnect * 1e9)

        # Intra-group compute (tensor parallel)
        expert_flops = 2 * 3 * self.config.hidden_dim * self.config.expert_dim
        flops_per_gpu = tokens_per_expert * experts_per_group * expert_flops / gpus_per_group
        compute_time = flops_per_gpu / (self.compute * 1e12)

        # Intra-group all-reduce
        intragroup_bytes = tokens * self.config.hidden_dim * self.config.param_bytes / expert_groups
        intragroup_time = 2 * intragroup_bytes * (gpus_per_group - 1) / gpus_per_group / (self.interconnect * 1e9)

        total_time = compute_time + intergroup_time + intragroup_time

        return {
            'strategy': f'hybrid_{expert_groups}groups',
            'total_time_ms': total_time * 1000,
            'compute_time_ms': compute_time * 1000,
            'comm_time_ms': (intergroup_time + intragroup_time) * 1000,
            'memory_per_gpu_gb': self.config.total_size_gb / self.num_gpus,
        }


# Compare strategies
sim = ExpertParallelismSimulator(models['DeepSeek-V3'], num_gpus=8)

batch_sizes = [1, 4, 16, 64, 256]
print(f"Expert Parallelism Comparison (DeepSeek-V3, 8 GPUs):")
print(f"{'BS':>5} {'Strategy':>20} {'Total (ms)':>12} {'Compute':>10} {'Comm':>10}")
print("-" * 60)
for bs in [1, 16, 64]:
    for strat_fn in [sim.expert_parallel, sim.tensor_parallel]:
        r = strat_fn(bs, 1)  # Single decode step
        print(f"{bs:>5} {r['strategy']:>20} {r['total_time_ms']:>10.3f}ms "
              f"{r['compute_time_ms']:>8.3f}ms {r['comm_time_ms']:>8.3f}ms")
    r = sim.hybrid_parallel(bs, 1, expert_groups=2)
    print(f"{bs:>5} {r['strategy']:>20} {r['total_time_ms']:>10.3f}ms "
          f"{r['compute_time_ms']:>8.3f}ms {r['comm_time_ms']:>8.3f}ms")
    print()

In [ ]:
# Visualize parallelism strategies across batch sizes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

batch_range = [1, 2, 4, 8, 16, 32, 64, 128, 256]
for strat_name, strat_fn in [('Expert Parallel', sim.expert_parallel),
                               ('Tensor Parallel', sim.tensor_parallel)]:
    times = [strat_fn(bs, 1)['total_time_ms'] for bs in batch_range]
    ax1.plot(batch_range, times, 'o-', label=strat_name, linewidth=2)

hybrid_times = [sim.hybrid_parallel(bs, 1, 2)['total_time_ms'] for bs in batch_range]
ax1.plot(batch_range, hybrid_times, 'o-', label='Hybrid (2 groups)', linewidth=2)

ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Time per Step (ms)')
ax1.set_title('MoE Layer Time vs Batch Size')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log', base=2)

# Compute vs communication breakdown
for strat_name, strat_fn in [('Expert Parallel', sim.expert_parallel),
                               ('Tensor Parallel', sim.tensor_parallel)]:
    comp_frac = [strat_fn(bs, 1)['compute_time_ms'] / strat_fn(bs, 1)['total_time_ms']
                 for bs in batch_range]
    ax2.plot(batch_range, comp_frac, 'o-', label=strat_name, linewidth=2)

ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Compute Fraction')
ax2.set_title('Compute vs Communication Balance')
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log', base=2)

plt.tight_layout()
plt.savefig('moe_parallelism.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Assignment 1: Build an Expert Load Balancer

In [ ]:
# ASSIGNMENT 1: Expert Load Balancer

class ExpertLoadBalancer:
    """
    TODO: Implement a load balancer that:
    1. Monitors expert utilization in real-time
    2. Redistributes experts across GPUs when imbalance detected
    3. Implements auxiliary loss for balanced routing
    4. Handles capacity limits per expert
    """

    def __init__(self, num_experts: int, num_gpus: int,
                 capacity_factor: float = 1.25):
        self.num_experts = num_experts
        self.num_gpus = num_gpus
        self.capacity_factor = capacity_factor

        # Expert-to-GPU assignment
        self.assignment = {}
        experts_per_gpu = num_experts // num_gpus
        for i in range(num_experts):
            self.assignment[i] = i // experts_per_gpu

        # Utilization tracking
        self.expert_counts = np.zeros(num_experts)
        self.gpu_loads = np.zeros(num_gpus)
        self.rebalance_history = []

    def update_counts(self, expert_activations: np.ndarray):
        """
        TODO: Update expert activation counts and GPU loads.
        """
        # YOUR CODE HERE
        for eid in expert_activations.flatten():
            self.expert_counts[eid] += 1
            gpu = self.assignment[eid]
            self.gpu_loads[gpu] += 1

    def compute_imbalance(self) -> float:
        """
        TODO: Compute GPU load imbalance metric.
        Returns ratio of max/mean load (1.0 = perfect balance).
        """
        # YOUR CODE HERE
        if np.mean(self.gpu_loads) == 0:
            return 1.0
        return np.max(self.gpu_loads) / np.mean(self.gpu_loads)

    def rebalance(self) -> dict:
        """
        TODO: Rebalance experts across GPUs to minimize load imbalance.
        Strategy: Move popular experts to less loaded GPUs.
        """
        # YOUR CODE HERE
        # Sort experts by popularity
        sorted_experts = np.argsort(self.expert_counts)[::-1]

        # Round-robin assignment of popular experts
        new_assignment = {}
        gpu_new_loads = np.zeros(self.num_gpus)

        for i, eid in enumerate(sorted_experts):
            # Assign to least loaded GPU
            target_gpu = int(np.argmin(gpu_new_loads))
            new_assignment[eid] = target_gpu
            gpu_new_loads[target_gpu] += self.expert_counts[eid]

        moves = sum(1 for e in range(self.num_experts)
                    if new_assignment[e] != self.assignment.get(e, -1))

        old_imbalance = self.compute_imbalance()
        self.assignment = new_assignment
        # Recalculate GPU loads
        self.gpu_loads = np.zeros(self.num_gpus)
        for eid, gpu in self.assignment.items():
            self.gpu_loads[gpu] += self.expert_counts[eid]
        new_imbalance = self.compute_imbalance()

        result = {
            'moves': moves,
            'old_imbalance': old_imbalance,
            'new_imbalance': new_imbalance,
            'improvement': old_imbalance - new_imbalance,
        }
        self.rebalance_history.append(result)
        return result


# Test load balancer
balancer = ExpertLoadBalancer(num_experts=64, num_gpus=8)

# Simulate skewed workload
router = ExpertRouter(64, 4, 'skewed')
indices = router.route(10000)
balancer.update_counts(indices)

print(f"Before rebalancing:")
print(f"  GPU loads: {balancer.gpu_loads}")
print(f"  Imbalance: {balancer.compute_imbalance():.2f}x")

result = balancer.rebalance()
print(f"\nAfter rebalancing:")
print(f"  GPU loads: {balancer.gpu_loads}")
print(f"  Imbalance: {result['new_imbalance']:.2f}x")
print(f"  Expert moves: {result['moves']}")
print(f"  Improvement: {result['improvement']:.2f}")

---
## Assignment 2: Implement Expert Offloading

In [ ]:
# ASSIGNMENT 2: Expert Offloading

class ExpertOffloader:
    """
    TODO: Implement expert offloading where:
    1. Only popular experts stay on GPU
    2. Less popular experts are loaded on-demand from CPU
    3. Prefetching predicts next needed experts
    """

    def __init__(self, num_experts: int, gpu_capacity: int,
                 expert_size_gb: float,
                 pcie_bandwidth_gbps: float = 64.0):
        self.num_experts = num_experts
        self.gpu_capacity = gpu_capacity
        self.expert_size = expert_size_gb
        self.bandwidth = pcie_bandwidth_gbps

        self.gpu_experts = set(range(min(gpu_capacity, num_experts)))
        self.popularity = np.zeros(num_experts)
        self.prefetch_queue = set()

        self.gpu_hits = 0
        self.prefetch_hits = 0
        self.cpu_loads = 0
        self.total_overhead_s = 0

    def process_batch(self, expert_ids: List[int]) -> dict:
        """
        TODO: Process a batch of expert requests.

        Returns:
        - overhead_ms: loading overhead
        - loaded: number of experts loaded from CPU
        """
        # YOUR CODE HERE
        overhead = 0
        loaded = 0

        unique_experts = set(expert_ids)
        for eid in unique_experts:
            self.popularity[eid] += 1

            if eid in self.gpu_experts:
                self.gpu_hits += 1
            elif eid in self.prefetch_queue:
                self.prefetch_hits += 1
                overhead += self.expert_size / self.bandwidth * 0.3  # Partial load
                self.gpu_experts.add(eid)
                loaded += 1
            else:
                self.cpu_loads += 1
                overhead += self.expert_size / self.bandwidth
                self.gpu_experts.add(eid)
                loaded += 1

        # Evict if over capacity
        while len(self.gpu_experts) > self.gpu_capacity:
            # Evict least popular
            least_pop = min(self.gpu_experts, key=lambda e: self.popularity[e])
            self.gpu_experts.discard(least_pop)

        # Prefetch: predict next experts based on popularity
        self.prefetch_queue = set()
        sorted_by_pop = np.argsort(self.popularity)[::-1]
        for eid in sorted_by_pop[:self.gpu_capacity * 2]:
            if eid not in self.gpu_experts:
                self.prefetch_queue.add(eid)
                if len(self.prefetch_queue) >= 5:
                    break

        self.total_overhead_s += overhead
        return {'overhead_ms': overhead * 1000, 'loaded': loaded}

    def stats(self):
        total = self.gpu_hits + self.prefetch_hits + self.cpu_loads
        return {
            'gpu_hit_rate': self.gpu_hits / total if total > 0 else 0,
            'prefetch_hit_rate': self.prefetch_hits / total if total > 0 else 0,
            'cpu_load_rate': self.cpu_loads / total if total > 0 else 0,
            'total_overhead_s': self.total_overhead_s,
        }


# Test offloading with DeepSeek-V3
offloader = ExpertOffloader(
    num_experts=256, gpu_capacity=64,
    expert_size_gb=0.15,  # ~150MB per expert
)

router = ExpertRouter(256, 8, 'learned')
for batch in range(1000):
    indices = router.route(32)  # 32 tokens per batch
    offloader.process_batch(indices.flatten().tolist())

stats = offloader.stats()
print(f"Expert Offloading Results:")
for k, v in stats.items():
    if 'rate' in k:
        print(f"  {k}: {v:.1%}")
    else:
        print(f"  {k}: {v:.2f}")

---
## Assignment 3: Optimize MoE Throughput

In [ ]:
# ASSIGNMENT 3: MoE Throughput Optimization

class MoEThroughputOptimizer:
    """
    TODO: Optimize MoE serving throughput by:
    1. Batching tokens for same expert together
    2. Overlapping computation and communication
    3. Choosing optimal parallelism strategy
    4. Dynamic batch sizing based on expert activation patterns
    """

    def __init__(self, config: MoEConfig, num_gpus: int):
        self.config = config
        self.num_gpus = num_gpus
        self.sim = ExpertParallelismSimulator(config, num_gpus)

    def optimize_batch_schedule(self, token_expert_assignments: np.ndarray) -> dict:
        """
        TODO: Reorder tokens to maximize expert batch efficiency.

        Group tokens assigned to the same expert to enable
        larger, more efficient GEMM operations.
        """
        # YOUR CODE HERE
        num_tokens = len(token_expert_assignments)

        # Count tokens per expert
        expert_tokens = defaultdict(list)
        for token_idx, experts in enumerate(token_expert_assignments):
            for eid in experts:
                expert_tokens[eid].append(token_idx)

        # Compute batch sizes per expert
        batch_sizes = {eid: len(tokens) for eid, tokens in expert_tokens.items()}
        avg_batch = np.mean(list(batch_sizes.values()))
        max_batch = np.max(list(batch_sizes.values()))
        min_batch = np.min(list(batch_sizes.values()))

        # Compute efficiency: larger batches are more efficient for GEMM
        efficiency = sum(
            min(1.0, bs / 32) * bs  # Efficiency scales until batch=32
            for bs in batch_sizes.values()
        ) / sum(batch_sizes.values())

        return {
            'num_active_experts': len(expert_tokens),
            'avg_batch_per_expert': avg_batch,
            'max_batch': max_batch,
            'min_batch': min_batch,
            'compute_efficiency': efficiency,
            'expert_batch_distribution': batch_sizes,
        }

    def find_optimal_config(self, target_throughput: float,
                             max_latency_ms: float) -> dict:
        """
        TODO: Find optimal serving configuration.

        Search over:
        - Parallelism strategy
        - Batch size
        - Number of GPUs
        """
        # YOUR CODE HERE
        best = None
        for bs in [1, 4, 8, 16, 32, 64, 128]:
            for strat in ['expert', 'tensor']:
                if strat == 'expert':
                    r = self.sim.expert_parallel(bs, 1)
                else:
                    r = self.sim.tensor_parallel(bs, 1)

                latency = r['total_time_ms']
                throughput = bs / (latency / 1000)

                if latency <= max_latency_ms:
                    if best is None or throughput > best['throughput']:
                        best = {
                            'strategy': strat,
                            'batch_size': bs,
                            'latency_ms': latency,
                            'throughput': throughput,
                        }

        return best or {'strategy': 'none', 'batch_size': 1, 'latency_ms': 0, 'throughput': 0}


# Test optimization
optimizer = MoEThroughputOptimizer(models['DeepSeek-V3'], num_gpus=8)

# Generate workload and optimize
router = ExpertRouter(256, 8, 'learned')
assignments = router.route(1024)
schedule = optimizer.optimize_batch_schedule(assignments)

print(f"Batch Schedule Optimization:")
print(f"  Active experts: {schedule['num_active_experts']}")
print(f"  Avg batch per expert: {schedule['avg_batch_per_expert']:.1f}")
print(f"  Max/Min batch: {schedule['max_batch']}/{schedule['min_batch']}")
print(f"  Compute efficiency: {schedule['compute_efficiency']:.2%}")

# Find optimal config
optimal = optimizer.find_optimal_config(target_throughput=1000, max_latency_ms=50)
print(f"\nOptimal Config (target 1000 tok/s, max 50ms latency):")
for k, v in optimal.items():
    print(f"  {k}: {v}")

## Summary

### Key Takeaways

1. **Expert load imbalance** is a fundamental challenge -- real routing is heavily skewed with top 10% experts handling 30-50% of tokens

2. **Expert caching** enables serving models with hundreds of experts on limited GPU memory, achieving 95%+ hit rates with proper sizing

3. **Expert parallelism** is generally better than tensor parallelism for MoE at small batch sizes due to lower communication overhead

4. **Load rebalancing** by redistributing experts across GPUs based on popularity can reduce imbalance by 30-50%

5. **Token batching per expert** improves GEMM efficiency significantly -- grouping tokens for the same expert enables larger, more efficient matrix multiplications